# CYR-GPU-002 — Cymek GPU research campaign (preregistered)

Run cells in order: **CELL 0 → CELL 1 → CELL 2**. Use a **GPU runtime**
(Runtime → Change runtime type → GPU). Target 120–160 min, hard stop 175.
Do not edit thresholds, seeds, budgets, or arms: they are preregistered
and hash-bound (CELL 0 aborts on any mismatch). Every shell command
below fails hard: a nonzero exit stops the cell, never prints COMPLETE.

In [ ]:
# CELL 0 — frozen commit checkout, SHA verification, env, validation gate
import hashlib, json, subprocess, sys
from pathlib import Path

REPO = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
BRANCH = "cymek-500m-readiness"
WORK = Path("/content/An-Ra-the-new-AGI")
PREREG_REL = "docs/cymek/experiments/CYR-GPU-002/PREREGISTRATION.json"

def run_checked(argv, **kw):
    completed = subprocess.run(argv, capture_output=True, text=True, **kw)
    print((completed.stdout + completed.stderr)[-1500:])
    assert completed.returncode == 0, f"command failed: {argv[0]}"
    return completed

if not (WORK / ".git").exists():
    run_checked(["git", "clone", "--branch", BRANCH, REPO, str(WORK)])
import os
os.chdir(str(WORK))
bootstrap = json.loads((WORK / PREREG_REL).read_text())
pinned = bootstrap["cymek_head_sha"]
run_checked(["git", "fetch", "origin", pinned])
run_checked(["git", "checkout", pinned])
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                      text=True, check=True).stdout.strip()
print("HEAD:", head)
prereg = json.loads(Path(PREREG_REL).read_text())
assert head == prereg["cymek_head_sha"], "HEAD is not the preregistered commit"
for rel, expected in prereg["code_sha256"].items():
    actual = hashlib.sha256(Path(rel).read_bytes()).hexdigest()
    assert actual == expected, f"preregistered code mismatch: {rel}"
    print("OK", rel)
print("prereg", prereg["sha256"][:12], "experiment", prereg["experiment_id"])

run_checked([sys.executable, "-m", "pip", "install", "-q",
             "pytest", "tokenizers", "psutil"])
import torch
assert torch.cuda.is_available(), "no GPU runtime: choose Runtime → GPU"
props = torch.cuda.get_device_properties(0)
print("gpu:", props.name, "vram_gb:", round(props.total_memory / 2**30, 1),
      "torch:", torch.__version__, "bf16:", props.major >= 8)

run_checked([sys.executable, "-m", "pytest",
             "tests/test_v5_cyr_tournament.py", "tests/test_v5_contracts.py",
             "tests/test_v5_complete_answer.py", "tests/test_v5_training.py",
             "tests/test_v5_data.py", "-q"])
run_checked([sys.executable, "-m", "anra_v5.cyr_execute", "--mode", "smoke",
             "--out", "/content/CYR-SMOKE"])
run_checked([sys.executable, "-m", "pytest", "tests/test_production_entry.py",
             "-q", "--deselect",
             "tests/test_production_entry.py::test_exact_head_test_receipt"])
print("CELL 0 COMPLETE: validation gate green")


In [ ]:
# CELL 1 — full preregistered campaign (~120-160 min, hard stop 175)
import os, subprocess, sys
from pathlib import Path
os.environ["COLAB_GPU"] = "1"
os.chdir("/content/An-Ra-the-new-AGI")
mirror = "/content/drive/MyDrive/AnRa/CYR-GPU-002"
mirror_args = ["--mirror", mirror] if Path("/content/drive").exists() else []
if not Path("/content/drive").exists():
    print("WARNING: Drive not mounted; storage is EPHEMERAL_STORAGE_ONLY")
cmd = [sys.executable, "-m", "anra_v5.cyr_execute", "--mode", "full",
       "--prereg", "docs/cymek/experiments/CYR-GPU-002/PREREGISTRATION.json",
       "--repo", ".", "--out", "/content/CYR-GPU-002",
       "--stages", "s0,s1,s2,s3,s4"] + mirror_args
completed = subprocess.run(cmd)
assert completed.returncode == 0, "campaign failed; see FAILURE.json in output dir"
print("CELL 1 COMPLETE")


In [ ]:
# CELL 2 — verify bundle, print decision, download ZIP
import json
from pathlib import Path
out = Path("/content/CYR-GPU-002")
manifest = json.loads((out / "SESSION_MANIFEST.json").read_text())
print("stages:", manifest.get("stages", manifest))
decision = json.loads((out / "DECISION.json").read_text())
print(json.dumps({k: decision[k] for k in ("experiment", "stages", "gates", "claim_ladder") if k in decision}, indent=2))
print("promotion_bar:", json.dumps(decision.get("promotion_bar", {}), indent=2))
zip_path = out / "CYMEK_GPU_RESEARCH_V2_RESULTS.zip"
assert zip_path.is_file(), "result bundle missing"
from google.colab import files
files.download(str(zip_path))
print("Return CYMEK_GPU_RESEARCH_V2_RESULTS.zip to the operator.")
